# Capstone ICS2122 — Análisis de Demanda


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

demanda = pd.read_csv("demanda_historica.csv")
red = pd.read_csv("demanda_red_diaria.csv")
hospitales = pd.read_csv("hospitales.csv")

demanda["fecha"] = pd.to_datetime(demanda["fecha"])
red["fecha"] = pd.to_datetime(red["fecha"])

print("Archivos cargados correctamente.")

## 1. Calidad de los datos

Primero verificamos que la base sea completa y consistente antes de analizarla.


In [ ]:
# 1. CALIDAD DE LOS DATOS

print("Período:", demanda["fecha"].min().date(), "a", demanda["fecha"].max().date())
print("Hospitales:", demanda["hospital"].nunique())
print("Días:", demanda["fecha"].nunique())
print("Observaciones:", len(demanda))
print("Nulos en demanda:", demanda["demanda"].isna().sum())
print("Demandas negativas:", (demanda["demanda"] < 0).sum())
print("Duplicados hospital-fecha:", demanda.duplicated(["hospital", "fecha"]).sum())

# Verificar que la suma de hospitales coincida con demanda_red_diaria
red_calculada = (
    demanda.groupby("fecha", as_index=False)["demanda"]
    .sum()
    .rename(columns={"demanda": "demanda_total_calculada"})
)

validacion = red.merge(red_calculada, on="fecha")
validacion["diferencia"] = (
    validacion["demanda_total_red"] - validacion["demanda_total_calculada"]
)

print("Máxima diferencia con demanda_red_diaria:",
      validacion["diferencia"].abs().max())

## 1.1 Analisis demanda total diaria

In [ ]:
# Estadísticas descriptivas
promedio = red["demanda_total_red"].mean()
desv_std = red["demanda_total_red"].std()
minimo = red["demanda_total_red"].min()
maximo = red["demanda_total_red"].max()
q1 = red["demanda_total_red"].quantile(0.25)
q3 = red["demanda_total_red"].quantile(0.75)

print(f"Promedio demanda total diaria de la red: {promedio:.1f} unidades/día ")
print(f"Desviación estándar: {desv_std:.1f}")
print(f"Mínimo histórico de: {minimo:.0f} ")
print(f"Máximo de: {maximo:.0f} unidades/día")
print(f"El 50% de los días se ubica entre: {q1:.0f} y {q3:.0f} unidades/día.")

In [ ]:
# 1.1 HISTOGRAMA DE LA DEMANDA TOTAL DIARIA

plt.figure(figsize=(10, 6))
plt.hist(red["demanda_total_red"], bins=30, edgecolor="black")

plt.title("Distribución de la demanda total diaria")
plt.xlabel("Demanda total")
plt.ylabel("Frecuencia")

plt.tight_layout()
plt.show()

## 2. Diferencias entre hospitales


In [ ]:
# 2. ESTADÍSTICOS POR HOSPITAL

stats = (
    demanda.groupby("hospital")["demanda"]
    .agg(
        promedio="mean",
        mediana="median",
        desviacion_std="std",
        minimo="min",
        maximo="max"
    )
    .reset_index()
)

p95 = (
    demanda.groupby("hospital")["demanda"]
    .quantile(0.95)
    .rename("p95")
    .reset_index()
)

stats = stats.merge(p95, on="hospital")
stats["cv"] = stats["desviacion_std"] / stats["promedio"]

stats = stats.sort_values("promedio", ascending=False)

display(stats.round(2))


In [ ]:
# 2.1 HOSPITALES CON MAYOR DEMANDA Y MAYOR VARIABILIDAD

print("Mayor demanda promedio:")
display(stats.nlargest(5, "promedio")[["hospital", "promedio", "p95", "cv"]].round(2))

print("Mayor variabilidad relativa:")
display(stats.nlargest(5, "cv")[["hospital", "promedio", "p95", "cv"]].round(2))


In [ ]:
# 2.2 GRÁFICO: DEMANDA PROMEDIO POR HOSPITAL

orden = stats.sort_values("promedio")

plt.figure(figsize=(10, 7))
plt.barh(orden["hospital"], orden["promedio"])
plt.xlabel("Demanda promedio diaria")
plt.ylabel("Hospital")
plt.title("Demanda promedio por hospital")
plt.tight_layout()
plt.show()


In [ ]:
# 2.3 GRÁFICO: PROMEDIO VS VARIABILIDAD

plt.figure(figsize=(8, 6))
plt.scatter(stats["promedio"], stats["cv"])

for _, fila in stats.nlargest(3, "cv").iterrows():
    plt.annotate(
        fila["hospital"],
        (fila["promedio"], fila["cv"]),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel("Demanda promedio diaria")
plt.ylabel("Coeficiente de variación (CV)")
plt.title("Demanda promedio vs variabilidad")
plt.tight_layout()
plt.show()

In [ ]:
# 2.4 BOXPLOT DE DEMANDA DIARIA POR HOSPITAL

orden_hospitales = (
    demanda.groupby("hospital")["demanda"]
    .mean()
    .sort_values(ascending=True)
    .index
)

datos_boxplot = [
    demanda.loc[demanda["hospital"] == h, "demanda"]
    for h in orden_hospitales
]

plt.figure(figsize=(12, 8))
plt.boxplot(
    datos_boxplot,
    vert=False,
    tick_labels=orden_hospitales,
    showfliers=False
)

plt.title("Distribución de demanda diaria por hospital")
plt.xlabel("Demanda diaria")
plt.ylabel("Hospital")

plt.tight_layout()
plt.show()

## 3. Demanda versus capacidad

In [ ]:
# 3. COBERTURA DE INVENTARIO

hosp = hospitales[hospitales["tipo"] == "Hospital"][
    ["nombre", "tamano", "capacidad_maxima"]
].rename(columns={"nombre": "hospital"})

cobertura = stats.merge(hosp, on="hospital", how="left")

cobertura["cobertura_media_dias"] = (
    cobertura["capacidad_maxima"] / cobertura["promedio"]
)
cobertura["cobertura_p95_dias"] = (
    cobertura["capacidad_maxima"] / cobertura["p95"]
)

display(
    cobertura[[
        "hospital", "capacidad_maxima", "promedio", "p95",
        "cobertura_media_dias", "cobertura_p95_dias"
    ]]
    .sort_values("cobertura_p95_dias")
    .round(2)
)

In [ ]:
# 3.1 HOSPITALES CON MENOR COBERTURA BAJO DEMANDA ALTA

display(
    cobertura.nsmallest(7, "cobertura_p95_dias")[[
        "hospital", "promedio", "p95",
        "capacidad_maxima", "cobertura_p95_dias"
    ]].round(2)
)

In [ ]:
# 3.2 GRÁFICO: COBERTURA P95 POR HOSPITAL

grafico_cobertura = cobertura.sort_values("cobertura_p95_dias")

plt.figure(figsize=(10, 8))

plt.barh(
    grafico_cobertura["hospital"],
    grafico_cobertura["cobertura_p95_dias"]
)

plt.axvline(
    x=2,
    linestyle="--",
    linewidth=2,
    label="Umbral de 2 días"
)

plt.xlabel("Días de cobertura con demanda P95")
plt.ylabel("Hospital")
plt.title("Cobertura de inventario bajo demanda alta (P95)")

plt.legend()

plt.tight_layout()
plt.show()

## 4. Comportamiento temporal

Revisamos si existe un patrón claro (estacionalidad) 

In [ ]:
# 4. DEMANDA TOTAL POR DÍA DE SEMANA Y MES

red_temp = red.copy()
red_temp["dia_semana_num"] = red_temp["fecha"].dt.dayofweek
red_temp["mes"] = red_temp["fecha"].dt.month

dias = {
    0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves",
    4: "Viernes", 5: "Sábado", 6: "Domingo"
}
red_temp["dia_semana"] = red_temp["dia_semana_num"].map(dias)

por_dia = (
    red_temp.groupby(["dia_semana_num", "dia_semana"])["demanda_total_red"]
    .mean()
    .reset_index()
    .sort_values("dia_semana_num")
)

por_mes = (
    red_temp.groupby("mes")["demanda_total_red"]
    .mean()
    .reset_index()
)

print("Promedio por día de semana:")
display(por_dia.round(2))

print("Promedio por mes:")
display(por_mes.round(2))


In [ ]:
# 4.1 GRÁFICOS TEMPORALES

plt.figure(figsize=(8, 4))
plt.plot(por_dia["dia_semana"], por_dia["demanda_total_red"], marker="o")
plt.ylabel("Demanda promedio")
plt.title("Demanda total promedio por día de semana")
plt.xticks(rotation=30)

max_dia = por_dia["demanda_total_red"].max()
plt.ylim(0, max_dia * 1.10)

plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 4))
plt.plot(por_mes["mes"], por_mes["demanda_total_red"], marker="o")
plt.xlabel("Mes")
plt.ylabel("Demanda promedio")
plt.title("Demanda total promedio por mes")
plt.xticks(range(1, 13))

max_mes = por_mes["demanda_total_red"].max()
plt.ylim(0, max_mes * 1.10)

plt.tight_layout()
plt.show()

In [ ]:
# 4.2 DEMANDA TOTAL PROMEDIO POR AÑO

red_anio = red.copy()
red_anio["anio"] = red_anio["fecha"].dt.year

promedio_anual = (
    red_anio.groupby("anio")["demanda_total_red"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.plot(
    promedio_anual["anio"],
    promedio_anual["demanda_total_red"],
    marker="o"
)

plt.title("Demanda total promedio por año")
plt.xlabel("Año")
plt.ylabel("Demanda promedio")

plt.xticks(promedio_anual["anio"])

# Eje Y desde 0 con 10% de margen superior
max_anual = promedio_anual["demanda_total_red"].max()
plt.ylim(0, max_anual * 1.10)

plt.tight_layout()
plt.show()

## 5. Dependencia temporal

Comprobamos si la demanda de un día está fuertemente relacionada con la demanda de días anteriores.

Nos interesa principalmente:

- lag 1: día anterior;
- lag 7: misma posición de la semana anterior.


In [ ]:
# 5. AUTOCORRELACIÓN DE LA DEMANDA TOTAL

acf_1 = red["demanda_total_red"].autocorr(lag=1)
acf_7 = red["demanda_total_red"].autocorr(lag=7)

print("Autocorrelación lag 1:", round(acf_1, 3))
print("Autocorrelación lag 7:", round(acf_7, 3))

## 5.2. Dependencia temporal por hospital

El análisis anterior de autocorrelación se realizó sobre la demanda total de la red. Ahora estudiamos cada hospital individualmente.

Queremos saber si la demanda reciente de un hospital entrega información sobre su demanda futura.

Calculamos autocorrelaciones para rezagos de 1, 2, 3, 5, 7, 14, 21 y 28 días.

In [ ]:
# 5.2. AUTOCORRELACIÓN POR HOSPITAL

lags = [1, 2, 3, 5, 7, 14, 21, 28]

resultados_acf = []

for hospital, grupo in demanda.groupby("hospital"):

    serie = (
        grupo.sort_values("fecha")
        .set_index("fecha")["demanda"]
    )

    fila = {
        "hospital": hospital
    }

    for lag in lags:
        fila[f"lag_{lag}"] = serie.autocorr(lag=lag)

    resultados_acf.append(fila)


acf_hospital = pd.DataFrame(resultados_acf)

columnas_lag = [f"lag_{lag}" for lag in lags]

acf_hospital["max_acf_abs"] = (
    acf_hospital[columnas_lag]
    .abs()
    .max(axis=1)
)

display(
    acf_hospital.sort_values(
        "max_acf_abs",
        ascending=False
    ).round(3)
)

In [ ]:
# 5.2.1 HOSPITALES CON MAYOR DEPENDENCIA TEMPORAL

grafico_acf = (
    acf_hospital.sort_values(
        "max_acf_abs",
        ascending=True
    )
)

plt.figure(figsize=(10, 8))

plt.barh(
    grafico_acf["hospital"],
    grafico_acf["max_acf_abs"]
)

plt.xlabel("Máxima autocorrelación absoluta")
plt.ylabel("Hospital")
plt.title("Máxima dependencia temporal observada por hospital")

plt.tight_layout()
plt.show()

In [ ]:
# 5.2.2 LAG CON MAYOR AUTOCORRELACIÓN POR HOSPITAL

resumen_acf = acf_hospital[
    ["hospital"] + columnas_lag
].copy()

resumen_acf["lag_mayor_dependencia"] = (
    resumen_acf[columnas_lag]
    .abs()
    .idxmax(axis=1)
    .str.replace("lag_", "")
    .astype(int)
)

resumen_acf["acf_mayor_dependencia"] = resumen_acf.apply(
    lambda fila: fila[
        f"lag_{int(fila['lag_mayor_dependencia'])}"
    ],
    axis=1
)

display(
    resumen_acf[
        [
            "hospital",
            "lag_mayor_dependencia",
            "acf_mayor_dependencia"
        ]
    ]
    .sort_values(
        "acf_mayor_dependencia",
        key=abs,
        ascending=False
    )
    .round(3)
)

## 5.3. Volatility clustering y persistencia de episodios de alta demanda

Una autocorrelación baja en la demanda no implica necesariamente que la incertidumbre sea independiente en el tiempo.

Por ello analizamos si días con desviaciones grandes respecto del comportamiento normal tienden a ser seguidos por otros días también inusualmente variables.

Además, definimos como "día de demanda alta" aquel cuya demanda supera el percentil 90 histórico del hospital y estudiamos si haber observado días altos recientemente modifica la probabilidad de una nueva demanda alta.

In [ ]:
# 5.3. VOLATILITY CLUSTERING

demanda_vol = demanda.sort_values(
    ["hospital", "fecha"]
).copy()

media_hospital = (
    demanda_vol.groupby("hospital")["demanda"]
    .transform("mean")
)

demanda_vol["residuo"] = (
    demanda_vol["demanda"] - media_hospital
)

demanda_vol["residuo_abs"] = demanda_vol["residuo"].abs()
demanda_vol["residuo_cuadrado"] = demanda_vol["residuo"] ** 2

lags_vol = [1, 2, 3, 5, 7]

resultados_vol = []

for hospital, grupo in demanda_vol.groupby("hospital"):

    grupo = grupo.sort_values("fecha")

    fila = {
        "hospital": hospital
    }

    for lag in lags_vol:

        fila[f"vol_abs_lag_{lag}"] = (
            grupo["residuo_abs"]
            .autocorr(lag=lag)
        )

        fila[f"vol2_lag_{lag}"] = (
            grupo["residuo_cuadrado"]
            .autocorr(lag=lag)
        )

    resultados_vol.append(fila)


volatilidad_acf = pd.DataFrame(resultados_vol)

columnas_vol = [
    f"vol2_lag_{lag}"
    for lag in lags_vol
]

volatilidad_acf["max_acf_volatilidad"] = (
    volatilidad_acf[columnas_vol]
    .abs()
    .max(axis=1)
)

display(
    volatilidad_acf.sort_values(
        "max_acf_volatilidad",
        ascending=False
    ).round(3)
)

## 6. Eventos extremos

Detectamos valores extremos usando el criterio IQR.

**No los eliminamos automáticamente**, porque en este contexto pueden corresponder a episodios reales de alta demanda.


In [ ]:
# 6. OUTLIERS POR HOSPITAL

resumen_outliers = []

for hospital, grupo in demanda.groupby("hospital"):
    q1 = grupo["demanda"].quantile(0.25)
    q3 = grupo["demanda"].quantile(0.75)
    iqr = q3 - q1
    limite_superior = q3 + 1.5 * iqr

    outliers = grupo[grupo["demanda"] > limite_superior]

    resumen_outliers.append({
        "hospital": hospital,
        "n_outliers": len(outliers),
        "maximo": grupo["demanda"].max(),
        "limite_superior_iqr": limite_superior
    })

resumen_outliers = pd.DataFrame(resumen_outliers)

cv_hospital = (
    demanda.groupby("hospital")["demanda"]
    .agg(["mean", "std"])
    .reset_index()
)

cv_hospital["cv"] = cv_hospital["std"] / cv_hospital["mean"]

tabla_outliers = resumen_outliers.merge(
    cv_hospital[["hospital", "cv"]],
    on="hospital"
)

tabla_outliers = tabla_outliers[
    ["hospital", "cv", "n_outliers", "maximo", "limite_superior_iqr"]
].sort_values(
    "n_outliers",
    ascending=False
)

tabla_outliers = tabla_outliers.rename(columns={
    "hospital": "Hospital",
    "cv": "CV",
    "n_outliers": "N° eventos extremos",
    "maximo": "Máximo",
    "limite_superior_iqr": "Límite superior IQR"
})

display(tabla_outliers.round(2))

## 7. Segmentación operacional simple

No usamos clustering automático.

Creamos indicadores fáciles de interpretar:

- **Alta demanda:** hospital sobre el percentil 75 de demanda promedio.
- **Alta variabilidad:** CV mayor o igual a 0,30.
- **Baja cobertura:** menos de 2 días de cobertura usando P95.

Esto permite identificar hospitales especialmente exigentes para la política de distribución.


In [ ]:
# 7. INDICADORES OPERACIONALES

resultado = cobertura.copy()

umbral_demanda_alta = resultado["promedio"].quantile(0.75)

resultado["alta_demanda"] = resultado["promedio"] >= umbral_demanda_alta
resultado["alta_variabilidad"] = resultado["cv"] >= 0.30
resultado["baja_cobertura"] = resultado["cobertura_p95_dias"] < 2

resultado["n_factores_criticos"] = (
    resultado[["alta_demanda", "alta_variabilidad", "baja_cobertura"]]
    .sum(axis=1)
)

display(
    resultado[[
        "hospital", "promedio", "cv", "p95",
        "cobertura_p95_dias",
        "alta_demanda", "alta_variabilidad",
        "baja_cobertura", "n_factores_criticos"
    ]]
    .sort_values(
        ["n_factores_criticos", "promedio"],
        ascending=[False, False]
    )
    .round(2)
)

In [ ]:
# 7.1 TABLA DE FACTORES OPERACIONALES CRÍTICOS

tabla_criticos = resultado[
    [
        "hospital",
        "promedio",
        "cv",
        "cobertura_p95_dias",
        "alta_demanda",
        "alta_variabilidad",
        "baja_cobertura",
        "n_factores_criticos"
    ]
].copy()

tabla_criticos = tabla_criticos[
    tabla_criticos["n_factores_criticos"] > 0
]

for col in ["alta_demanda", "alta_variabilidad", "baja_cobertura"]:
    tabla_criticos[col] = tabla_criticos[col].map({
        True: "X",
        False: ""
    })

tabla_criticos = tabla_criticos.rename(
    columns={
        "hospital": "Hospital",
        "promedio": "Promedio",
        "cv": "CV",
        "cobertura_p95_dias": "Cobertura P95 (días)",
        "alta_demanda": "Alta demanda",
        "alta_variabilidad": "Alta variabilidad",
        "baja_cobertura": "Baja cobertura",
        "n_factores_criticos": "N° factores críticos"
    }
)

tabla_criticos = tabla_criticos.sort_values(
    ["N° factores críticos", "Promedio"],
    ascending=[False, False]
)

display(tabla_criticos.round(2))

## 8. Demanda acumulada según horizonte de abastecimiento

Para evaluar posibles frecuencias de visita, se analiza la demanda acumulada entre
abastecimientos consecutivos para horizontes de 1, 2, 3, 5 y 7 días.

Para cada hospital se calcula la proporción histórica de ventanas en que la demanda
acumulada supera su capacidad. Como referencia exploratoria, se considera crítico un
horizonte cuando esta proporción supera el 5%.

Este indicador no representa directamente una probabilidad de quiebre de stock, ya que
todavía no incorpora inventario inicial ni reposiciones.

In [ ]:
# 8.1 DEMANDA ACUMULADA SEGÚN HORIZONTE

horizontes = [1, 2, 3, 5, 7]
umbral_riesgo = 0.05

capacidades = (
    hospitales[hospitales["tipo"] == "Hospital"]
    [["nombre", "capacidad_maxima"]]
    .rename(columns={"nombre": "hospital"})
)

resultados_horizonte = []

for hospital, grupo in demanda.groupby("hospital"):

    grupo = grupo.sort_values("fecha").copy()
    capacidad = capacidades.loc[
        capacidades["hospital"] == hospital,
        "capacidad_maxima"
    ].iloc[0]

    for h in horizontes:

        acumulada = (
            grupo["demanda"]
            .rolling(window=h)
            .sum()
            .dropna()
        )

        resultados_horizonte.append({
            "hospital": hospital,
            "horizonte": h,
            "p95_acumulado": acumulada.quantile(0.95),
            "prop_supera_capacidad": (acumulada > capacidad).mean()
        })

resultados_horizonte = pd.DataFrame(resultados_horizonte)

In [ ]:
# 8.2 RESUMEN OPERACIONAL POR HORIZONTE

tabla_riesgo = (
    resultados_horizonte
    .pivot(
        index="hospital",
        columns="horizonte",
        values="prop_supera_capacidad"
    )
    .reset_index()
)

tabla_riesgo.columns = [
    "Hospital", "Riesgo 1 día", "Riesgo 2 días",
    "Riesgo 3 días", "Riesgo 5 días", "Riesgo 7 días"
]

primer_critico = (
    resultados_horizonte[
        resultados_horizonte["prop_supera_capacidad"] > 0.05
    ]
    .groupby("hospital")["horizonte"]
    .min()
    .rename("Primer horizonte crítico")
    .reset_index()
    .rename(columns={"hospital": "Hospital"})
)

tabla_resumen = tabla_riesgo.merge(
    primer_critico,
    on="Hospital",
    how="left"
)

tabla_resumen = tabla_resumen.sort_values(
    ["Primer horizonte crítico", "Riesgo 2 días"],
    ascending=[True, False]
)

columnas_riesgo = [
    "Riesgo 1 día", "Riesgo 2 días", "Riesgo 3 días",
    "Riesgo 5 días", "Riesgo 7 días"
]

tabla_resumen[columnas_riesgo] = (
    tabla_resumen[columnas_riesgo] * 100
).round(1)

display(tabla_resumen)

criticos_horizonte = (
    resultados_horizonte
    .assign(
        critico=lambda x: x["prop_supera_capacidad"] > 0.05
    )
    .groupby("horizonte")["critico"]
    .sum()
)

plt.figure(figsize=(7, 4))
plt.bar(criticos_horizonte.index, criticos_horizonte.values)

plt.xlabel("Días entre abastecimientos")
plt.ylabel("N° de hospitales con riesgo > 5%")
plt.title("Hospitales críticos según horizonte de abastecimiento")
plt.xticks(horizontes)
plt.ylim(0, 22)

plt.tight_layout()
plt.show()

## 9. Dependencia entre hospitales

Se evaluó si las demandas de distintos hospitales tienden a comportarse conjuntamente.

Se utilizó la correlación de Pearson para detectar dependencia lineal entre los niveles
de demanda y lift para evaluar si episodios de demanda alta (sobre el P90 de cada
hospital) ocurren simultáneamente con mayor frecuencia que la esperada.

Este análisis permite evaluar si la dependencia entre hospitales debería considerarse
al construir escenarios conjuntos de demanda.

In [ ]:
# 9.1 DEPENDENCIA ENTRE HOSPITALES

from itertools import combinations

wide = demanda.pivot(
    index="fecha",
    columns="hospital",
    values="demanda"
)

p90 = wide.quantile(0.90)
demanda_alta = wide.gt(p90)

resultados_dependencia = []

for h1, h2 in combinations(wide.columns, 2):

    correlacion = wide[h1].corr(wide[h2])

    p1 = demanda_alta[h1].mean()
    p2 = demanda_alta[h2].mean()

    ambos_altos = demanda_alta[h1] & demanda_alta[h2]
    p_ambos = ambos_altos.mean()

    lift = p_ambos / (p1 * p2)

    resultados_dependencia.append({
        "Hospital 1": h1,
        "Hospital 2": h2,
        "Correlación": correlacion,
        "Lift": lift,
        "Días ambos altos": ambos_altos.sum()
    })

dependencia = pd.DataFrame(resultados_dependencia)

dependencia["Correlación absoluta"] = (
    dependencia["Correlación"].abs()
)

In [ ]:
# 9.2 RESULTADOS PRINCIPALES

top_correlacion = (
    dependencia
    .sort_values("Correlación absoluta", ascending=False)
    .head(5)
    [["Hospital 1", "Hospital 2", "Correlación"]]
)

print("Pares con mayor correlación absoluta:")
display(top_correlacion.round(3))

top_lift = (
    dependencia
    .sort_values("Lift", ascending=False)
    .head(5)
    [["Hospital 1", "Hospital 2", "Lift", "Días ambos altos"]]
)

print("Pares con mayor coincidencia relativa de demanda alta:")
display(top_lift.round(3))

plt.figure(figsize=(7, 5))

plt.scatter(
    dependencia["Correlación"],
    dependencia["Lift"],
    alpha=0.6
)

plt.axvline(0, linestyle="--")
plt.axhline(1, linestyle="--")

plt.xlabel("Correlación de Pearson")
plt.ylabel("Lift de demanda alta")
plt.title("Dependencia entre pares de hospitales")

plt.tight_layout()
plt.show()

## 10. Distribución probabilística de la demanda por hospital

Para construir posteriormente escenarios estocásticos de demanda, se evalúa qué
distribuciones probabilísticas representan razonablemente el comportamiento histórico
de cada hospital.

El objetivo no es encontrar un ajuste perfecto, sino determinar si una familia
probabilística sencilla puede reproducir adecuadamente el nivel, la dispersión y,
especialmente, la cola alta de la demanda observada.

In [ ]:
# 10.1 AJUSTE PROBABILÍSTICO POR HOSPITAL

import numpy as np
import pandas as pd

from scipy.stats import poisson, nbinom, binom
from scipy.optimize import minimize_scalar


resultados_modelos = []

for hospital, grupo in demanda.groupby("hospital"):

    x = grupo["demanda"].dropna().to_numpy()

    # La demanda debe ser discreta
    if not np.allclose(x, np.round(x)):
        raise ValueError(f"La demanda de {hospital} contiene valores no enteros.")

    x = np.round(x).astype(int)

    media = x.mean()
    varianza = x.var(ddof=1)
    var_media = varianza / media
    p95_obs = np.quantile(x, 0.95)

    candidatos = []


    # ---------------------------
    # 1. POISSON
    # ---------------------------
    lambda_p = media

    ll_poisson = poisson.logpmf(x, mu=lambda_p).sum()
    aic_poisson = 2 * 1 - 2 * ll_poisson
    p95_poisson = poisson.ppf(0.95, mu=lambda_p)

    candidatos.append({
        "modelo": "Poisson",
        "aic": aic_poisson,
        "p95_modelo": p95_poisson
    })


    # ---------------------------
    # 2. BINOMIAL NEGATIVA
    # Solo tiene sentido si varianza > media
    # ---------------------------
    if varianza > media:

        def negloglik_nb(log_r):
            r = np.exp(log_r)
            p = r / (r + media)

            return -nbinom.logpmf(
                x,
                n=r,
                p=p
            ).sum()

        ajuste_nb = minimize_scalar(
            negloglik_nb,
            bounds=(-8, 12),
            method="bounded"
        )

        r = np.exp(ajuste_nb.x)
        p = r / (r + media)

        ll_nb = -ajuste_nb.fun
        aic_nb = 2 * 2 - 2 * ll_nb
        p95_nb = nbinom.ppf(0.95, n=r, p=p)

        candidatos.append({
            "modelo": "Binomial negativa",
            "aic": aic_nb,
            "p95_modelo": p95_nb
        })


    # ---------------------------
    # 3. BINOMIAL
    # Solo tiene sentido si varianza < media
    # ---------------------------
    if varianza < media:

        n_min = max(
            int(x.max()),
            int(np.ceil(media))
        )

        n_max = max(
            n_min + 100,
            int(n_min * 4)
        )

        mejor_ll = -np.inf
        mejor_n = None
        mejor_p = None

        for n in range(n_min, n_max + 1):

            p = media / n

            if 0 < p < 1:

                ll = binom.logpmf(
                    x,
                    n=n,
                    p=p
                ).sum()

                if ll > mejor_ll:
                    mejor_ll = ll
                    mejor_n = n
                    mejor_p = p

        if mejor_n is not None:

            aic_binom = 2 * 2 - 2 * mejor_ll
            p95_binom = binom.ppf(
                0.95,
                n=mejor_n,
                p=mejor_p
            )

            candidatos.append({
                "modelo": "Binomial",
                "aic": aic_binom,
                "p95_modelo": p95_binom
            })


    # Elegir modelo con menor AIC
    mejor = min(candidatos, key=lambda z: z["aic"])

    resultados_modelos.append({
        "Hospital": hospital,
        "Media": media,
        "Varianza": varianza,
        "Var/Media": var_media,
        "Mejor modelo": mejor["modelo"],
        "AIC mejor": mejor["aic"],
        "P95 observado": p95_obs,
        "P95 modelo": mejor["p95_modelo"],
        "Error P95": abs(mejor["p95_modelo"] - p95_obs)
    })


tabla_modelos = pd.DataFrame(resultados_modelos)

In [ ]:
# 10.2 RESUMEN DE DISTRIBUCIONES SELECCIONADAS

tabla_resumen_modelos = tabla_modelos[
    [
        "Hospital",
        "Var/Media",
        "Mejor modelo",
        "P95 observado",
        "P95 modelo",
        "Error P95"
    ]
].copy()

# Ordenar según mayor error en la cola alta
tabla_resumen_modelos = tabla_resumen_modelos.sort_values(
    "Error P95",
    ascending=False
)

display(tabla_resumen_modelos.round(2))


# Resumen general
print("\nNúmero de hospitales según modelo seleccionado:")
display(
    tabla_modelos["Mejor modelo"]
    .value_counts()
    .rename("N° hospitales")
    .to_frame()
)

print(
    "Error promedio en P95:",
    round(tabla_modelos["Error P95"].mean(), 2),
    "unidades"
)

print(
    "Error máximo en P95:",
    round(tabla_modelos["Error P95"].max(), 2),
    "unidades"
)

In [ ]:
# 10.3 EJEMPLOS DE AJUSTE PROBABILÍSTICO

from scipy.stats import poisson, nbinom, binom
from scipy.optimize import minimize_scalar

hospitales_ejemplo = {
    "Hospital Calvo Mackenna": "Binomial negativa",
    "Hospital Luis Tisné": "Binomial",
    "Hospital Parroquial San Bernardo": "Poisson"
}

for hospital, modelo in hospitales_ejemplo.items():

    x = (
        demanda.loc[demanda["hospital"] == hospital, "demanda"]
        .dropna()
        .round()
        .astype(int)
        .to_numpy()
    )

    media = x.mean()
    varianza = x.var(ddof=1)

    # Valores posibles de demanda
    valores = np.arange(x.min(), x.max() + 1)

    # Frecuencia empírica
    frecuencia_observada = (
        pd.Series(x)
        .value_counts(normalize=True)
        .reindex(valores, fill_value=0)
        .values
    )

    # -------------------------
    # Ajuste según distribución
    # -------------------------

    if modelo == "Poisson":

        prob_modelo = poisson.pmf(
            valores,
            mu=media
        )

    elif modelo == "Binomial negativa":

        def negloglik_nb(log_r):
            r = np.exp(log_r)
            p = r / (r + media)

            return -nbinom.logpmf(
                x,
                n=r,
                p=p
            ).sum()

        ajuste = minimize_scalar(
            negloglik_nb,
            bounds=(-8, 12),
            method="bounded"
        )

        r = np.exp(ajuste.x)
        p = r / (r + media)

        prob_modelo = nbinom.pmf(
            valores,
            n=r,
            p=p
        )

    elif modelo == "Binomial":

        n_min = max(
            int(x.max()),
            int(np.ceil(media))
        )

        n_max = max(
            n_min + 100,
            int(n_min * 4)
        )

        mejor_ll = -np.inf

        for n in range(n_min, n_max + 1):

            p = media / n

            if 0 < p < 1:

                ll = binom.logpmf(
                    x,
                    n=n,
                    p=p
                ).sum()

                if ll > mejor_ll:
                    mejor_ll = ll
                    mejor_n = n
                    mejor_p = p

        prob_modelo = binom.pmf(
            valores,
            n=mejor_n,
            p=mejor_p
        )

    # -------------------------
    # Gráfico
    # -------------------------

    plt.figure(figsize=(8, 4))

    plt.bar(
        valores,
        frecuencia_observada,
        alpha=0.6,
        label="Demanda observada"
    )

    plt.plot(
        valores,
        prob_modelo,
        marker="o",
        markersize=3,
        label=f"Ajuste {modelo}"
    )

    plt.title(f"{hospital} – {modelo}")
    plt.xlabel("Demanda diaria")
    plt.ylabel("Probabilidad")
    plt.legend()

    plt.tight_layout()
    plt.show()